# Quandela Swaptions Challenge - Working Notebook

This notebook is adapted to this repo layout and auto-detects the dataset folder inside `quandela_folder`.


In [ ]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.linear_model import Ridge
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import StandardScaler

np.random.seed(0)
pd.set_option('display.max_columns', 20)

cwd = Path.cwd().resolve()
PROJECT_ROOT = cwd.parent if cwd.name.lower() == 'notebooks' else cwd

candidate_dirs = [
    PROJECT_ROOT / 'Quandela_folder' / 'CHALLENGE RESOURCES' / 'DATASETS',
    PROJECT_ROOT / 'quandela_folder' / 'CHALLENGE RESOURCES' / 'DATASETS',
    PROJECT_ROOT,
]

def find_file(name: str) -> Path:
    for d in candidate_dirs:
        p = d / name
        if p.exists():
            return p
    raise FileNotFoundError(f"Could not find {name}. Checked: {[str(d) for d in candidate_dirs]}")

TRAIN_XLSX = find_file('train.xlsx')
TEMPLATE_XLSX = find_file('test_template.xlsx')
OUTPUT_DIR = PROJECT_ROOT / 'outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('PROJECT_ROOT:', PROJECT_ROOT)
print('TRAIN_XLSX:', TRAIN_XLSX)
print('TEMPLATE_XLSX:', TEMPLATE_XLSX)
print('OUTPUT_DIR:', OUTPUT_DIR)


In [ ]:
train_df = pd.read_excel(TRAIN_XLSX)
train_df['Date'] = pd.to_datetime(train_df['Date'], dayfirst=True)
train_df = train_df.sort_values('Date').reset_index(drop=True)

surface_cols = [c for c in train_df.columns if c != 'Date']
assert len(surface_cols) == 224, f'Expected 224 surface columns, got {len(surface_cols)}'
assert train_df[surface_cols].isna().sum().sum() == 0, 'Train contains NaNs.'

print('train shape:', train_df.shape)
print('date range:', train_df['Date'].min().date(), '->', train_df['Date'].max().date())
print('surface columns:', len(surface_cols))
train_df.head(2)


In [ ]:
def parse_tenor_maturity(col: str):
    # Supports labels like 'Tenor: 1; Maturity: 0.5' and fallback numeric extraction.
    parts = [p.strip() for p in col.split(';')]
    if len(parts) >= 2 and ':' in parts[0] and ':' in parts[1]:
        t = float(parts[0].split(':', 1)[1].strip())
        m = float(parts[1].split(':', 1)[1].strip())
        return t, m
    nums = re.findall(r"[-+]?(?:\d*\.\d+|\d+)", col)
    if len(nums) < 2:
        raise ValueError(f'Cannot parse tenor/maturity from column: {col}')
    return float(nums[0]), float(nums[1])

pairs = [parse_tenor_maturity(c) for c in surface_cols]
tenors = sorted({t for t, _ in pairs})
maturities = sorted({m for _, m in pairs})
assert len(tenors) * len(maturities) == 224

print('grid:', len(tenors), 'x', len(maturities))
print('tenor sample:', tenors[:5])
print('maturity sample:', maturities[:5])


In [ ]:
Y = train_df[surface_cols].values.astype(float)
scaler = StandardScaler()
Yz = scaler.fit_transform(Y)

k = min(12, Y.shape[1], max(2, Y.shape[0] // 20))
pca = PCA(n_components=k, random_state=0)
F = pca.fit_transform(Yz)
print('PCA k =', k, '| explained variance =', float(pca.explained_variance_ratio_.sum()))

def make_window_dataset(F, W: int, H: int):
    T, _ = F.shape
    X_list, y_list = [], []
    for t in range(W - 1, T - H):
        X_list.append(F[t - W + 1:t + 1].reshape(-1))
        y_list.append(F[t + H])
    if not X_list:
        raise ValueError('Not enough rows for selected window/horizon.')
    return np.vstack(X_list), np.vstack(y_list)

W = min(20, max(5, len(F) // 8))
H = 1
X1, y1 = make_window_dataset(F, W=W, H=H)
print('window dataset:', X1.shape, y1.shape, '| W =', W)

def cv_score_ridge(X, y, alphas=(1e-3, 1e-2, 1e-1, 1, 10, 100), n_splits=5):
    n_splits = min(n_splits, max(2, len(X) - 1))
    tscv = TimeSeriesSplit(n_splits=n_splits)
    best = None
    for a in alphas:
        mses = []
        for tr, va in tscv.split(X):
            model = Ridge(alpha=a)
            model.fit(X[tr], y[tr])
            pred = model.predict(X[va])
            mses.append(mean_squared_error(y[va], pred))
        mse = float(np.mean(mses))
        if best is None or mse < best[0]:
            best = (mse, a)
    return best

best_mse, best_alpha = cv_score_ridge(X1, y1)
ridge = Ridge(alpha=best_alpha)
ridge.fit(X1, y1)
print('best alpha:', best_alpha, '| cv mse:', best_mse)

def predict_factors_iterative(F_hist, steps: int, W: int, ridge_model):
    F_ext = F_hist.copy()
    for _ in range(steps):
        x = F_ext[-W:].reshape(1, -1)
        f_next = ridge_model.predict(x)
        F_ext = np.vstack([F_ext, f_next])
    return F_ext[-steps:]

def factors_to_surface(F_fut):
    Yz_fut = pca.inverse_transform(F_fut)
    return scaler.inverse_transform(Yz_fut)


In [ ]:
template_df = pd.read_excel(TEMPLATE_XLSX)
tmpl_surface_cols = [c for c in template_df.columns if c not in ('Type', 'Date')]

missing_cols = set(surface_cols) - set(tmpl_surface_cols)
if missing_cols:
    raise ValueError(f'Template is missing {len(missing_cols)} columns. Example: {list(missing_cols)[:3]}')

n_rows = len(template_df)
F_future = predict_factors_iterative(F, steps=n_rows, W=W, ridge_model=ridge)
Y_future = factors_to_surface(F_future)

pred_df = pd.DataFrame(Y_future, columns=surface_cols)[tmpl_surface_cols]
submission_df = template_df.copy()
submission_df.loc[:, tmpl_surface_cols] = pred_df.values

submission_path = OUTPUT_DIR / 'submission_baseline.xlsx'
submission_df.to_excel(submission_path, index=False)
print('submission saved to:', submission_path)
submission_df.head(2)


## Visualization
Quick plots to inspect the training surfaces and baseline forecast output.


In [ ]:
import matplotlib.pyplot as plt

def column_for(tenor_target: float, maturity_target: float):
    best_col = None
    best_dist = float("inf")
    for c in surface_cols:
        t, m = parse_tenor_maturity(c)
        d = abs(t - tenor_target) + abs(m - maturity_target)
        if d < best_dist:
            best_dist = d
            best_col = c
    return best_col

series_specs = [(1, 1), (5, 5), (10, 10), (5, 1)]
fig, axes = plt.subplots(len(series_specs), 1, figsize=(12, 10), sharex=True)
for ax, (tt, mm) in zip(axes, series_specs):
    col = column_for(tt, mm)
    ax.plot(train_df["Date"], train_df[col], lw=1.5)
    ax.set_title(f"{col}")
    ax.set_ylabel("Price")
axes[-1].set_xlabel("Date")
plt.tight_layout()
plt.show()


In [ ]:
pair_map = {c: parse_tenor_maturity(c) for c in surface_cols}
tenor_idx = {t: i for i, t in enumerate(sorted({tm[0] for tm in pair_map.values()}))}
maturity_idx = {m: j for j, m in enumerate(sorted({tm[1] for tm in pair_map.values()}))}

def row_to_surface(row_values):
    surf = np.full((len(tenor_idx), len(maturity_idx)), np.nan)
    for c in surface_cols:
        t, m = pair_map[c]
        surf[tenor_idx[t], maturity_idx[m]] = row_values[c]
    return surf

last_surface = row_to_surface(train_df.iloc[-1])
plt.figure(figsize=(11, 6))
im = plt.imshow(last_surface, aspect="auto", interpolation="nearest", cmap="viridis")
plt.title(f"Last Training Surface ({train_df.iloc[-1]['Date'].date()})")
plt.xlabel("Maturity index")
plt.ylabel("Tenor index")
plt.colorbar(im, label="Price")
plt.tight_layout()
plt.show()


In [ ]:
pred_surface = row_to_surface(pred_df.iloc[0])
vmin = min(np.nanmin(last_surface), np.nanmin(pred_surface))
vmax = max(np.nanmax(last_surface), np.nanmax(pred_surface))

fig, axs = plt.subplots(1, 2, figsize=(14, 5))
im0 = axs[0].imshow(last_surface, aspect="auto", interpolation="nearest", cmap="viridis", vmin=vmin, vmax=vmax)
axs[0].set_title("Last Train Surface")
axs[0].set_xlabel("Maturity index")
axs[0].set_ylabel("Tenor index")

im1 = axs[1].imshow(pred_surface, aspect="auto", interpolation="nearest", cmap="viridis", vmin=vmin, vmax=vmax)
axs[1].set_title("First Predicted Surface")
axs[1].set_xlabel("Maturity index")
axs[1].set_ylabel("Tenor index")

fig.colorbar(im1, ax=axs.ravel().tolist(), label="Price")
plt.tight_layout()
plt.show()


## Optional quantum scaffold
The next cell is optional and does not install packages automatically.


In [ ]:
HAS_QUANTUM = False
try:
    import torch
    import perceval as pcvl
    from merlin import QuantumLayer
    HAS_QUANTUM = True
except Exception as e:
    print('Quantum stack not available, skipping optional section:', repr(e))

if HAS_QUANTUM:
    kq = min(6, F.shape[1])
    Wq = min(10, W)
    print('Quantum section ready. kq =', kq, '| Wq =', Wq)
